In [ ]:
CREATE TABLE IF NOT EXISTS DMT_FCT_KB_PROCESSING_STATUS (
  KB_SYS_ID          STRING PRIMARY KEY,
  STATUS             STRING,
  SOURCE_UPDATED_ON  TIMESTAMP_NTZ,
  PROCESSED_AT       TIMESTAMP_NTZ,
  NUM_CHUNKS         INT,
  ERROR_MESSAGE      STRING,
  PROCESSING_VERSION STRING
)

In [ ]:
CREATE TABLE IF NOT EXISTS DMT_FCT_KB_CHUNKS (
  KB_SYS_ID          STRING,
  CHUNK_INDEX        INT,
  CHUNK_TEXT         STRING,
  SOURCE_UPDATED_ON  TIMESTAMP_NTZ,
  PROCESSING_VERSION STRING,
  CREATED_AT         TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
  CONSTRAINT PK_KB_CHUNK UNIQUE (KB_SYS_ID, CHUNK_INDEX)
)

In [ ]:
MERGE INTO DMT_FCT_KB_PROCESSING_STATUS t
USING (
  SELECT SYS_ID, SYS_UPDATED_ON
  FROM FOO.BAR.DMT_FCT_KB_KNOWLEDGE
  WHERE TEXT IS NOT NULL AND ACTIVE = TRUE AND LATEST = TRUE
) s
ON t.KB_SYS_ID = s.SYS_ID
WHEN NOT MATCHED THEN
  INSERT (KB_SYS_ID, STATUS, SOURCE_UPDATED_ON)
  VALUES (s.SYS_ID, 'PENDING', s.SYS_UPDATED_ON)
WHEN MATCHED AND s.SYS_UPDATED_ON > t.SOURCE_UPDATED_ON THEN
  UPDATE SET STATUS = 'PENDING', SOURCE_UPDATED_ON = s.SYS_UPDATED_ON;

SELECT STATUS, COUNT(*) as count FROM DMT_FCT_KB_PROCESSING_STATUS
GROUP BY STATUS ORDER BY STATUS;

In [ ]:
-- Test UDTF works
SELECT chunks.*
FROM (VALUES ('test', '<h1>Title</h1><p>Content here.</p>')) AS kb(id, text),
TABLE(CHUNK_HTML_VECTORIZED(kb.id, kb.text) OVER (PARTITION BY 1)) chunks

In [ ]:
CREATE OR REPLACE FUNCTION CHUNK_HTML_VECTORIZED(
    article_sys_id STRING,
    article_text STRING
)
RETURNS TABLE (
    KB_SYS_ID STRING,
    CHUNK_INDEX INT, 
    CHUNK_TEXT STRING, 
    ERROR_MSG STRING
)
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('pandas', 'langchain-text-splitters', 'bs4')
HANDLER = 'ChunkHandler'
AS
$$
import pandas as pd
from langchain_text_splitters import HTMLSemanticPreservingSplitter, RecursiveCharacterTextSplitter
from _snowflake import vectorized

class ChunkHandler:
    def __init__(self):
        self.splitter = HTMLSemanticPreservingSplitter(
            headers_to_split_on=[("h1", "Header 1"), ("h2", "Header 2")],
            # about 512 tokens, suitable for CSS
            max_chunk_size=2000, # about 512 tokens, suitable for CSS
            elements_to_preserve=["table", "ul", "ol", "code"],
            denylist_tags=["script", "style", "head"],
            preserve_links=True,
            preserve_images=True,
        )
        # Fallback splitter for content without semantic headers
        self.fallback_splitter = RecursiveCharacterTextSplitter(
            chunk_size=2000,
            chunk_overlap=200,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
    
    @vectorized(input=pd.DataFrame)
    def end_partition(self, df):
        # TODO: substitute any image tags in the HTML with their captions
        # Uses semantic splitter for structured HTML (with h1/h2 headers)
        # Falls back to RecursiveCharacterTextSplitter for unstructured HTML

        for sys_id, text in zip(df['ARTICLE_SYS_ID'], df['ARTICLE_TEXT']):
            if pd.isna(text) or not str(text).strip():
                yield pd.DataFrame([{
                    'KB_SYS_ID': sys_id, 
                    'CHUNK_INDEX': -1,
                    'CHUNK_TEXT': None, 
                    'ERROR_MSG': 'Empty text'
                }])
                continue
            
            try:
                chunks = self.splitter.split_text(str(text))
                
                # If semantic splitter produces no chunks, use fallback splitter
                if not chunks:
                    chunks = self.fallback_splitter.split_text(str(text))
                
                chunk_data = []
                for idx, doc in enumerate(chunks):
                    chunk_data.append({
                        'KB_SYS_ID': sys_id,
                        'CHUNK_INDEX': idx,
                        'CHUNK_TEXT': getattr(doc, 'page_content', doc),
                        'ERROR_MSG': None
                    })
                
                if chunk_data:
                    yield pd.DataFrame(chunk_data)
                else:
                    yield pd.DataFrame([{
                        'KB_SYS_ID': sys_id, 
                        'CHUNK_INDEX': -1,
                        'CHUNK_TEXT': None, 
                        'ERROR_MSG': 'No chunks'
                    }])
            except Exception as e:
                yield pd.DataFrame([{
                    'KB_SYS_ID': sys_id, 
                    'CHUNK_INDEX': -1,
                    'CHUNK_TEXT': None, 
                    'ERROR_MSG': str(e)[:500]
                }])
$$;

In [ ]:
CREATE OR REPLACE PROCEDURE PROCESS_KB_BATCH(BATCH_SIZE INT, VERSION STRING)
RETURNS STRING
LANGUAGE SQL
EXECUTE AS CALLER
AS
BEGIN
  -- Step 1: Run UDTF once and capture query ID
SELECT 
    chunks.KB_SYS_ID, 
    chunks.CHUNK_INDEX, 
    chunks.CHUNK_TEXT, 
    chunks.ERROR_MSG,
    kb.SYS_UPDATED_ON
  FROM (
    SELECT kb.SYS_ID, kb.TEXT, kb.SYS_UPDATED_ON
    FROM FOO.BAR.DMT_FCT_KB_KNOWLEDGE kb
    JOIN DMT_FCT_KB_PROCESSING_STATUS status ON kb.SYS_ID = status.KB_SYS_ID
    WHERE status.STATUS = 'PENDING' 
      AND kb.TEXT IS NOT NULL 
      AND kb.TEXT != ''
      AND kb.ACTIVE = TRUE 
      AND kb.LATEST = TRUE
    LIMIT :BATCH_SIZE
  ) kb,
  TABLE(CHUNK_HTML_VECTORIZED(kb.SYS_ID, kb.TEXT) OVER (PARTITION BY 1)) chunks;
  
  LET chunk_qid := LAST_QUERY_ID();
  
  -- Step 2: Delete existing chunks for articles being reprocessed (prevents duplicates)
  DELETE FROM FOO.BAR.DMT_FCT_KB_CHUNKS
  WHERE KB_SYS_ID IN (
    SELECT DISTINCT KB_SYS_ID 
    FROM TABLE(RESULT_SCAN(:chunk_qid))
  );
  
  -- Step 3: Insert successful chunks 
  INSERT INTO FOO.BAR.DMT_FCT_KB_CHUNKS (
    KB_SYS_ID, CHUNK_INDEX, CHUNK_TEXT, SOURCE_UPDATED_ON, PROCESSING_VERSION, CREATED_AT
  )
  SELECT 
    KB_SYS_ID, 
    CHUNK_INDEX, 
    CHUNK_TEXT, 
    SYS_UPDATED_ON, 
    :VERSION, 
    CURRENT_TIMESTAMP()
  FROM TABLE(RESULT_SCAN(:chunk_qid))
  WHERE ERROR_MSG IS NULL AND CHUNK_INDEX >= 0;
  
  LET rows_inserted := SQLROWCOUNT;
  
  -- Step 4: Update status with completion info (for successful articles)
  UPDATE DMT_FCT_KB_PROCESSING_STATUS t
  SET 
    STATUS = 'COMPLETED',
    PROCESSED_AT = CURRENT_TIMESTAMP(),
    PROCESSING_VERSION = :VERSION,
    NUM_CHUNKS = (SELECT COUNT(*) FROM FOO.BAR.DMT_FCT_KB_CHUNKS c WHERE c.KB_SYS_ID = t.KB_SYS_ID),
    ERROR_MESSAGE = NULL
  WHERE KB_SYS_ID IN (
    SELECT DISTINCT KB_SYS_ID 
    FROM TABLE(RESULT_SCAN(:chunk_qid))
    WHERE ERROR_MSG IS NULL AND CHUNK_INDEX >= 0
  );
  
  -- Step 5: Update status with error info (for failed articles)
  UPDATE DMT_FCT_KB_PROCESSING_STATUS t
  SET 
    STATUS = 'FAILED',
    PROCESSED_AT = CURRENT_TIMESTAMP(),
    PROCESSING_VERSION = :VERSION,
    NUM_CHUNKS = 0,
    ERROR_MESSAGE = (
      SELECT MAX(ERROR_MSG) 
      FROM TABLE(RESULT_SCAN(:chunk_qid)) 
      WHERE KB_SYS_ID = t.KB_SYS_ID AND ERROR_MSG IS NOT NULL
    )
  WHERE KB_SYS_ID IN (
    SELECT DISTINCT KB_SYS_ID 
    FROM TABLE(RESULT_SCAN(:chunk_qid))
    WHERE ERROR_MSG IS NOT NULL
  );
  
  LET rows_updated := SQLROWCOUNT;
  
  RETURN 'Inserted ' || rows_inserted || ' chunks, updated ' || rows_updated || ' failed articles';
END;

In [ ]:
# Process all pending articles 
import time

BATCH_SIZE = 10000
VERSION = "v1.0"

pending = session.sql("SELECT COUNT(*) as c FROM DMT_FCT_KB_PROCESSING_STATUS WHERE STATUS = 'PENDING'").collect()[0]['C']

if pending == 0:
    print("✓ No articles to process")
else:
    print(f"Processing {pending} articles in batches of {BATCH_SIZE}...\n")
    start = time.time()
    
    batch_num = 1
    while True:
        batch_start = time.time()
        
        # Call stored procedure to process one batch
        result = session.call("PROCESS_KB_BATCH", BATCH_SIZE, VERSION)
        
        # Check progress
        stats = session.sql("""
        SELECT 
          COUNT(CASE WHEN STATUS = 'PENDING' THEN 1 END) as pending,
          COUNT(CASE WHEN STATUS = 'COMPLETED' THEN 1 END) as completed,
          COUNT(CASE WHEN STATUS = 'FAILED' THEN 1 END) as failed
        FROM DMT_FCT_KB_PROCESSING_STATUS
        """).collect()[0]
        
        elapsed = time.time() - batch_start
        print(f"Batch {batch_num}: "
              f"{stats['COMPLETED']} completed, {stats['FAILED']} failed, {stats['PENDING']} pending "
              f"({elapsed:.1f}s) - {result}")
        
        if stats['PENDING'] == 0:
            break
        
        batch_num += 1
    
    total_time = time.time() - start
    final = session.sql("SELECT COUNT(CASE WHEN STATUS = 'COMPLETED' THEN 1 END) as c, SUM(NUM_CHUNKS) as chunks FROM DMT_FCT_KB_PROCESSING_STATUS").collect()[0]
    print(f"\n✓ Processed {final['C']} articles → {final['CHUNKS']} chunks in {total_time/60:.1f} min")


## View Results


In [ ]:
# Summary statistics
summary = session.sql("""
SELECT 
  STATUS,
  COUNT(*) as articles,
  SUM(NUM_CHUNKS) as chunks
FROM DMT_FCT_KB_PROCESSING_STATUS
GROUP BY STATUS
ORDER BY STATUS
""").to_pandas()

print(summary.to_string(index=False))

total = session.sql("SELECT COUNT(*) as cnt FROM DMT_FCT_KB_CHUNKS").collect()[0]['CNT']
print(f"\nTotal chunks: {total:,}")


In [ ]:
# View failures (if any)
failures = session.sql("""
SELECT KB_SYS_ID, ERROR_MESSAGE
FROM DMT_FCT_KB_PROCESSING_STATUS
WHERE STATUS = 'FAILED'
LIMIT 20
""").to_pandas()

if len(failures) > 0:
    print(f"Failed articles ({len(failures)}):")
    for _, row in failures.iterrows():
        print(f"  {row['KB_SYS_ID'][:20]}: {row['ERROR_MESSAGE']}")
else:
    print("✓ No failures")


In [ ]:
# Option 1: Retry failed articles
session.sql("""
UPDATE DMT_FCT_KB_PROCESSING_STATUS
SET STATUS = 'PENDING', ERROR_MESSAGE = NULL
WHERE STATUS = 'FAILED'
""").collect()

cnt = session.sql("SELECT COUNT(*) as c FROM DMT_FCT_KB_PROCESSING_STATUS WHERE STATUS = 'PENDING'").collect()[0]['C']
print(f"✓ Reset {cnt} failed articles to PENDING")


In [ ]:
# Option 2: Reprocess when chunker logic changes
NEW_VERSION = "v2.0"  # Increment when you change UDTF logic

session.sql(f"""
UPDATE DMT_FCT_KB_PROCESSING_STATUS
SET STATUS = 'PENDING'
WHERE STATUS = 'COMPLETED'
  AND (PROCESSING_VERSION IS NULL OR PROCESSING_VERSION != '{NEW_VERSION}')
""").collect()

cnt = session.sql("SELECT COUNT(*) as c FROM DMT_FCT_KB_PROCESSING_STATUS WHERE STATUS = 'PENDING'").collect()[0]['C']
print(f"✓ Marked {cnt} articles for reprocessing (version {NEW_VERSION})")


In [ ]:
# Option 3: Reprocess articles older than X days (TTL)
TTL_DAYS = 30

session.sql(f"""
UPDATE DMT_FCT_KB_PROCESSING_STATUS
SET STATUS = 'PENDING'
WHERE STATUS = 'COMPLETED'
  AND PROCESSED_AT < DATEADD(day, -{TTL_DAYS}, CURRENT_TIMESTAMP())
""").collect()

cnt = session.sql("SELECT COUNT(*) as c FROM DMT_FCT_KB_PROCESSING_STATUS WHERE STATUS = 'PENDING'").collect()[0]['C']
print(f"✓ Marked {cnt} articles for reprocessing (>{TTL_DAYS} days old)")


In [ ]:
# Option 4: Reprocess specific articles
ARTICLE_SYS_IDS = ['sys_id_1', 'sys_id_2']  # Replace with actual SYS_IDs

if ARTICLE_SYS_IDS[0] != 'sys_id_1':
    ids = ",".join([f"'{x}'" for x in ARTICLE_SYS_IDS])
    session.sql(f"""
    UPDATE DMT_FCT_KB_PROCESSING_STATUS
    SET STATUS = 'PENDING'
    WHERE KB_SYS_ID IN ({ids})
    """).collect()
    print(f"✓ Marked {len(ARTICLE_SYS_IDS)} articles for reprocessing")
else:
    print("Update ARTICLE_SYS_IDS with actual values")


## Automation (Optional)


In [ ]:
# Create Snowflake Task to run automatically
CREATE OR REPLACE TASK TASK_PROCESS_KB_ARTICLES
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = 'USING CRON 0 */4 * * * UTC'  -- Every 4 hours
AS
BEGIN
  -- Sync tracking table
  MERGE INTO DMT_FCT_KB_PROCESSING_STATUS t
  USING (
    SELECT SYS_ID, SYS_UPDATED_ON
    FROM FOO.BAR.DMT_FCT_KB_KNOWLEDGE
    WHERE TEXT IS NOT NULL AND ACTIVE = TRUE AND LATEST = TRUE
  ) s
  ON t.KB_SYS_ID = s.SYS_ID
  WHEN NOT MATCHED THEN
    INSERT (KB_SYS_ID, STATUS, SOURCE_UPDATED_ON)
    VALUES (s.SYS_ID, 'PENDING', s.SYS_UPDATED_ON)
  WHEN MATCHED AND s.SYS_UPDATED_ON > t.SOURCE_UPDATED_ON THEN
    UPDATE SET STATUS = 'PENDING', SOURCE_UPDATED_ON = s.SYS_UPDATED_ON;
  
  -- Process pending batch (using stored procedure - runs UDTF once!)
  CALL PROCESS_KB_BATCH(500, 'v1.0');
END;

ALTER TASK TASK_PROCESS_KB_ARTICLES RESUME;